# Atomic Extraction — `Claude Opus 4.6`

Extracts all atomic fields from raw CV text in a single JSON prompt per CV.

**Fields:** `birth_year`, `birth_place`, `birth_country`, `sex`, `edu_start`, `edu_end`, `edu_degree`

## 1. Imports & config

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import json
import re
import time

import pandas as pd
from tqdm import tqdm

from corex_eval import evaluate, load_inputs
from corex_eval.gold import load_gold

import anthropic
client = anthropic.Anthropic()

MODEL_ID    = "claude-opus-4-6"
TEMPERATURE = 0
MAX_TOKENS  = 512

print(f"Model: {MODEL_ID}")

## 2. Load inputs

In [ ]:
inputs_df = load_inputs(task="extraction")
print(f"{len(inputs_df)} CVs in test set")
inputs_df.head()

## 3. Load valid codes for coded fields

In [ ]:
gold_df = load_gold()

# Valid codes for fields stored in codebook format
sex_codes     = sorted(gold_df["sex"].dropna().unique().tolist())
degree_codes  = sorted(gold_df["edu_degree"].dropna().unique().tolist())

sex_str     = "\n".join(f"  {c}" for c in sex_codes)
degree_str  = "\n".join(f"  {c}" for c in degree_codes)

print(f"sex codes ({len(sex_codes)}): {sex_codes}")
print(f"degree codes ({len(degree_codes)}): {degree_codes}")

## 4. Build system prompt

In [ ]:
SYSTEM_PROMPT = f"""You are an expert data extractor. Given a CV in any language, extract the following fields and return them as a single JSON object. If a field cannot be found, use null.

Fields to extract:
- birth_year    : integer year of birth (e.g. 1965)
- birth_place   : string, city or place of birth as written in the CV
- birth_country : free-text country name as you know it (e.g. Belgium, Germany), or null
- sex           : use EXACTLY one of the codes below, or null
{sex_str}
- edu_start     : integer year when highest education started
- edu_end       : integer year when highest education ended (or expected)
- edu_degree    : use EXACTLY one of the codes below, or null
{degree_str}

Respond with ONLY the JSON object — no explanation, no markdown fences.
Example: {{"birth_year": 1965, "birth_place": "Brussels", "birth_country": "17 = Belgium", "sex": "0 = male", "edu_start": 1983, "edu_end": 1988, "edu_degree": "7 = Master's or equivalent"}}
"""
print(f"System prompt: {len(SYSTEM_PROMPT)} chars")

## 5. Run inference

In [ ]:
def parse_json(text):
    """Extract JSON object from model response, tolerating markdown fences."""
    text = text.strip()
    text = re.sub(r'^```(?:json)?\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s*```$', '', text)
    m = re.search(r'\{[\s\S]*\}', text)
    if m:
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            pass
    return {}


rows = []

for _, row in tqdm(inputs_df.iterrows(), total=len(inputs_df), desc="Extracting"):
    case_id = str(row["case_id"])
    cv_text = str(row["cv_local"] or "")
    user_msg = f"CV text:\n\n{cv_text}"

    try:
        resp = client.messages.create(
            model=MODEL_ID,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_msg}],
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
        )
        raw = resp.content[0].text
    except Exception as e:
        print(f"Error on {case_id}: {e}")
        raw = "{}"
        time.sleep(1.0)

    parsed = parse_json(raw)
    rows.append({
        "case_id":       case_id,
        "birth_year":    parsed.get("birth_year"),
        "birth_place":   parsed.get("birth_place"),
        "birth_country": parsed.get("birth_country"),
        "sex":           parsed.get("sex"),
        "edu_start":     parsed.get("edu_start"),
        "edu_end":       parsed.get("edu_end"),
        "edu_degree":    parsed.get("edu_degree"),
    })

predictions_df = pd.DataFrame(rows)
predictions_df.head()

## 6. Evaluate & submit

In [ ]:
EXPERIMENT_PATH = "/home/tom/projects/corex_eval/experiments/extraction/claude_opus46_atomic/config.yaml"
ATOMIC_VARS = ["birth_year", "birth_place", "birth_country", "sex", "edu_start", "edu_end", "edu_degree"]

all_results = {}
for variable in ATOMIC_VARS:
    var_df = predictions_df[["case_id", variable]].copy()
    res = evaluate(
        var_df,
        task="extraction",
        variable=variable,
        submit=True,
        experiment_path=EXPERIMENT_PATH,
    )
    all_results[variable] = res
    print(f"{variable:15s}: accuracy={res.get('accuracy', res.get('mae', '?'))}")

pd.DataFrame({
    v: {k: r[k] for k in ('accuracy','mae','n_evaluated') if k in r}
    for v, r in all_results.items()
}).T

In [ ]:
from corex_eval.gold import load_gold

gold_df = load_gold()
gold_df["case_id"] = gold_df["case_id"].astype(str)

ATOMIC_VARS = ["birth_year", "birth_place", "birth_country", "sex", "edu_start", "edu_end", "edu_degree"]

compare_df = predictions_df.merge(
    gold_df[["case_id"] + ATOMIC_VARS],
    on="case_id",
    suffixes=("_pred", "_gold"),
)

# Interleave pred/gold columns per variable
cols = ["case_id"]
for v in ATOMIC_VARS:
    cols += [f"{v}_pred", f"{v}_gold"]
compare_df = compare_df[cols]

# Filter to mismatches only — set False to see all rows
mismatches_only = True
if mismatches_only:
    mask = False
    for v in ATOMIC_VARS:
        mask = mask | (compare_df[f"{v}_pred"].astype(str).str.lower().str.strip() !=
                       compare_df[f"{v}_gold"].astype(str).str.lower().str.strip())
    compare_df = compare_df[mask]

print(f"{len(compare_df)} rows with at least one mismatch")
compare_df.head(20)